# CP2 LastFM — Upload dan audit file di Colab

Notebook ini mengaudit file hasil olahan LastFM.txt dan LastFM_time.txt yang diunggah langsung ke sesi Colab. Google Drive maupun file lokal tidak diperlukan.

Batas audit: pemeriksaan mencakup format, alignment baris, jumlah nilai, gap, duplikasi item, dan ringkasan jumlah CP1. Tanpa file sumber HetRec atau kode preprocessing, audit belum membuktikan bahwa setiap timestamp cocok dengan event sumber yang benar.

Jalankan sel berurutan dari atas.


## 1. Upload dua file ke sesi Colab

Jalankan sel ini, lalu pilih kedua file dari komputer. Keduanya tersedia sementara di /content selama sesi Colab aktif.


In [ ]:
from google.colab import files
from pathlib import Path
uploaded = files.upload()
print("Uploaded:", list(uploaded.keys()))
ITEM_FILE = Path("/content/LastFM.txt")
TIME_FILE = Path("/content/LastFM_time.txt")
assert ITEM_FILE.exists(), "Upload file bernama LastFM.txt"
assert TIME_FILE.exists(), "Upload file bernama LastFM_time.txt"
print("Files ready:", ITEM_FILE, TIME_FILE)


## 2. Baca format sequence


In [ ]:
from collections import OrderedDict, Counter
import hashlib, json, statistics

def read_seq(path):
    rows=OrderedDict()
    for line_no,line in enumerate(path.read_text(encoding="utf-8-sig").splitlines(),1):
        parts=line.split()
        assert len(parts)>=2, f"Baris kosong/pendek: {path.name}:{line_no}"
        vals=list(map(int,parts)); uid,seq=vals[0],vals[1:]
        assert uid not in rows, f"ID user duplikat {uid} di {path.name}"
        rows[uid]=seq
    return rows
items=read_seq(ITEM_FILE)
times=read_seq(TIME_FILE)
print("Items: users",len(items),"events",sum(map(len,items.values())))
print("Time: users",len(times),"values",sum(map(len,times.values())))


## 3. Cek alignment, nilai waktu, dan item berulang


In [ ]:
assert list(items)==list(times), "Urutan atau ID user berbeda"
length_mismatch=[u for u in items if len(items[u])!=len(times[u])]
assert not length_mismatch, f"Jumlah item/waktu berbeda. Contoh user: {length_mismatch[:10]}"
negative=[(u,i,g) for u,seq in times.items() for i,g in enumerate(seq) if g<0]
first_not_zero=[u for u,seq in times.items() if not seq or seq[0]!=0]
# Anggap sementara file waktu berisi gap dalam detik. Nilai sangat besar ditandai untuk ditinjau, bukan dihapus.
large=[(u,i,g,g/86400) for u,seq in times.items() for i,g in enumerate(seq) if g>10*365*86400]
repeated=[(u,len(seq)-len(set(seq))) for u,seq in items.items() if len(seq)!=len(set(seq))]
all_times=[x for seq in times.values() for x in seq]
print("Alignment baris/panjang: PASS")
print("Negative values:",len(negative),"first value not zero:",len(first_not_zero))
print("Values above 10 years if seconds:",len(large),"examples:",large[:10])
print("Users with repeated artist IDs:",len(repeated),"repeated positions:",sum(n for _,n in repeated))
print("Time min/max:",min(all_times),max(all_times))
assert not negative, "Ada nilai waktu negatif; jangan lanjut sebelum format diperiksa"
assert not first_not_zero, "Jika file berisi gap, nilai pertama tiap user harus 0"


## 4. Bandingkan ringkasan dengan CP1

Ringkasan CP1 yang tercatat: 1.090 user, 3.646 artis, 52.551 interaksi. Angka cocok belum membuktikan sequence identik; ini hanya pemeriksaan awal.


In [ ]:
summary={
 "users":len(items),
 "artists":len({x for seq in items.values() for x in seq}),
 "interactions":sum(map(len,items.values())),
 "avg_sequence_length":statistics.mean(map(len,items.values())),
 "users_with_repeated_items":len(repeated),
 "repeated_item_positions":sum(n for _,n in repeated),
 "negative_time_values":len(negative),
 "large_values_if_seconds":len(large),
 "cp1_expected":{"users":1090,"artists":3646,"interactions":52551},
 "counts_match_cp1":len(items)==1090 and len({x for seq in items.values() for x in seq})==3646 and sum(map(len,items.values()))==52551,
 "sha256":{"LastFM.txt":hashlib.sha256(ITEM_FILE.read_bytes()).hexdigest(),"LastFM_time.txt":hashlib.sha256(TIME_FILE.read_bytes()).hexdigest()},
 "caveat":"Structural alignment does not prove per-event timestamp correctness. HetRec timestamps are tagging events, not listening/scrobble events."
}
print(json.dumps(summary,indent=2))


## 5. Simpan dan unduh laporan


In [ ]:
REPORT=Path("/content/LastFM_cp2_file_audit.json")
REPORT.write_text(json.dumps(summary,indent=2),encoding="utf-8")
from google.colab import files
files.download(str(REPORT))


## Cara melanjutkan

- Jika alignment, panjang sequence, nilai gap pertama, atau gap negatif gagal, berhenti dulu dan periksa file.
- Jika jumlah user/artis/interaksi berbeda dari CP1 atau ada artis berulang, preprocessing belum terbukti identik dengan CP1. Jangan klaim perbandingan satu-komponen.
- Nilai waktu lebih dari 10 tahun ditandai untuk ditinjau, bukan otomatis dibuang.
- Untuk bukti event-ke-sumber, upload juga file mentah HetRec dan audit bersama kode preprocessing.
- Dataset ini mencatat waktu pemberian tag artis, bukan waktu mendengarkan musik.
